# Run 1307 AI code from GCS in Colab

This notebook is meant to be opened from GitHub in Google Colab.
It syncs your `1307` code from a Google Cloud Storage bucket into `/content/1307`,
installs dependencies, checks GPU availability, and runs your entry script.

## 1) Configure

Fill in these values before running the rest of the notebook.

In [ ]:
GCP_PROJECT = "maloney-geog-473"
GCS_BUCKET = "gis-final-project"
GCS_PREFIX_1307 = "GIS Final Project/1307"

LOCAL_1307_DIR = "/content/1307"
REQUIREMENTS_FILE = "requirements.txt"  # relative to LOCAL_1307_DIR
EXTRA_PIP_PACKAGES = ""  # optional, space-separated

ENTRY_SCRIPT = "doe_geoai.py"
ENTRY_ARGS = ""  # leave blank to auto-build args for doe_geoai.py from DOE_* values below

# Optional dataset-build step (README workflow) via create_doe_dataset.py.
DOE_BUILD_DATASET = True
DOE_GRI_INPUT = "/content/1307/<your_actual_gri_file>.gri"  # REQUIRED
DOE_DATASET_OUT_DIR = "/content/doe-data/brady_samples_19x3d"
DOE_CHANNELS = 3
DOE_SAMPLE_COUNT = 100000
DOE_KERNEL_PIXELS = 19

# doe_geoai.py required/optional inputs.
DOE_DATASET_PATH = ""   # required: -d / --dataset (directory)
DOE_LABELBIN_PATH = ""  # optional override for -l; defaults under LOCAL_RUN_DIR when blank

# Optional output overrides (defaults to LOCAL_RUN_DIR if blank).
DOE_MODEL_PATH = ""   # -m / --model
DOE_PLOT_PATH = ""    # -p / --plot
DOE_CURVES_PATH = ""  # -o / --output_curves

# Common training knobs for auto-built doe_geoai.py args.
DOE_EPOCHS = 25
DOE_BATCH_SIZE = 32
DOE_GPUS = 1
DOE_EXTRA_ARGS = ""  # optional extra flags, e.g. "-a -v"

# Persistent run outputs in GCS.
GCS_OUTPUT_PREFIX = "GIS Final Project/outputs/1307"
RUN_NAME_OVERRIDE = ""  # leave blank for timestamped run names

# Optional: auto-append output args for scripts that support these flags.
AUTO_APPEND_OUTPUT_ARGS = False
OUTPUT_DIR_FLAG = "--output_dir"
SAVE_DIR_FLAG = "--save_dir"

## 2) Authenticate and sync from GCS

In [ ]:
import shlex
import subprocess
from pathlib import Path

from google.colab import auth

def run(cmd, cwd=None):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not GCP_PROJECT or not GCS_BUCKET:
    raise ValueError("Set GCP_PROJECT and GCS_BUCKET in the config cell first.")

auth.authenticate_user()
run(["gcloud", "config", "set", "project", GCP_PROJECT])

local_dir = Path(LOCAL_1307_DIR)
local_dir.mkdir(parents=True, exist_ok=True)

_prefix = GCS_PREFIX_1307.strip("/")
src = f"gs://{GCS_BUCKET}/{_prefix}" if _prefix else f"gs://{GCS_BUCKET}"
run(["gsutil", "-m", "rsync", "-r", src, str(local_dir)])

print("Synced to", local_dir)

## 3) Inspect synced files

In [ ]:
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing local code directory: {root}")

items = sorted(p.name for p in root.iterdir())
print(f"Top-level files/folders in {root}:")
for name in items[:200]:
    print(" -", name)

## 4) Install dependencies

In [ ]:
import sys
import shlex
from pathlib import Path

req_path = Path(LOCAL_1307_DIR) / REQUIREMENTS_FILE

run([sys.executable, "-m", "pip", "install", "-U", "pip"])
if req_path.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
else:
    print(f"No requirements file found at: {req_path}")

extra = EXTRA_PIP_PACKAGES.strip()
if extra:
    run([sys.executable, "-m", "pip", "install", *shlex.split(extra)])

## 5) Check GPU runtime

In [ ]:
import subprocess

try:
    run(["nvidia-smi"])
except subprocess.CalledProcessError:
    print("nvidia-smi failed. In Colab: Runtime -> Change runtime type -> GPU.")

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
except Exception as e:
    print("Torch check skipped:", e)

## 6) Create persistent run directories

This prepares a local run folder and a matching GCS destination so outputs can be synced off Colab runtime disk.

In [ ]:
from datetime import datetime
from pathlib import Path

run_name = RUN_NAME_OVERRIDE.strip() or datetime.now().strftime("run_%Y%m%d_%H%M%S")
LOCAL_RUN_DIR = Path(f"/content/1307_runs/{run_name}")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

_output_prefix = GCS_OUTPUT_PREFIX.strip("/")
if not _output_prefix:
    raise ValueError("Set GCS_OUTPUT_PREFIX in the config cell.")

GCS_RUN_URI = f"gs://{GCS_BUCKET}/{_output_prefix}/{run_name}"

print("Run name:", run_name)
print("Local run dir:", LOCAL_RUN_DIR)
print("GCS run uri:", GCS_RUN_URI)

## 7) (Optional) Build DOE dataset from GRI (README workflow)

Enable `DOE_BUILD_DATASET = True` and set `DOE_GRI_INPUT` to run `create_doe_dataset.py` before training.

This creates the dataset directory expected by `doe_geoai.py -d` and auto-sets `DOE_DATASET_PATH` to `DOE_DATASET_OUT_DIR`.

In [ ]:
import shlex
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing LOCAL_1307_DIR: {root}")

def find_matches(patterns):
    out = []
    for pat in patterns:
        out.extend(root.rglob(pat))
    files = sorted({str(p.resolve()) for p in out if p.is_file()})
    return files

if DOE_BUILD_DATASET:
    if not DOE_GRI_INPUT.strip():
        raise ValueError("Set DOE_GRI_INPUT when DOE_BUILD_DATASET=True")
    gri = Path(DOE_GRI_INPUT.strip())
    if not gri.is_file():
        raise FileNotFoundError(f"DOE_GRI_INPUT not found: {gri}")

    out_dir = Path(DOE_DATASET_OUT_DIR).resolve()
    out_dir.parent.mkdir(parents=True, exist_ok=True)

    build_cmd = [
        "python",
        str(root / "create_doe_dataset.py"),
        "-i", str(gri),
        "-c", str(DOE_CHANNELS),
        "-d", str(out_dir),
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    print("$", " ".join(shlex.quote(c) for c in build_cmd))
    run(build_cmd, cwd=str(root))

    DOE_DATASET_PATH = str(out_dir)
    print("Built dataset and set DOE_DATASET_PATH:", DOE_DATASET_PATH)
else:
    # Fallback auto-detection when dataset was prebuilt/uploaded.
    dataset_patterns = ["*samples*", "*dataset*", "*.h5", "*.hdf5", "*dataset*.npy", "*dataset*.npz"]
    labelbin_patterns = ["*label*bin*", "*label*.pickle", "*label*.pkl", "*label*.joblib", "*.l"]

    dataset_matches = find_matches(dataset_patterns)
    labelbin_matches = find_matches(labelbin_patterns)

    print("Dataset candidates:")
    for p in dataset_matches[:50]:
        print(" -", p)
    if not dataset_matches:
        print(" - none found")

    print("\nLabelbin candidates:")
    for p in labelbin_matches[:50]:
        print(" -", p)
    if not labelbin_matches:
        print(" - none found")

    if not DOE_DATASET_PATH.strip() and len(dataset_matches) == 1:
        DOE_DATASET_PATH = dataset_matches[0]
        print("\nAuto-set DOE_DATASET_PATH:", DOE_DATASET_PATH)
    elif not DOE_DATASET_PATH.strip() and len(dataset_matches) > 1:
        print("\nMultiple dataset candidates found; set DOE_DATASET_PATH manually in config.")

    if not DOE_LABELBIN_PATH.strip() and len(labelbin_matches) == 1:
        DOE_LABELBIN_PATH = labelbin_matches[0]
        print("Auto-set DOE_LABELBIN_PATH:", DOE_LABELBIN_PATH)
    elif not DOE_LABELBIN_PATH.strip() and len(labelbin_matches) > 1:
        print("Multiple labelbin candidates found; set DOE_LABELBIN_PATH manually in config.")

## 8) Write run metadata and execute your 1307 entry script

This stores the exact script/args/environment for reproducibility, then runs your job.

For `doe_geoai.py`, if `ENTRY_ARGS` is blank the notebook auto-builds required args from `DOE_DATASET_PATH` and output paths. Labels/model default under `LOCAL_RUN_DIR` when overrides are blank.

In [ ]:
import json
import platform
import shlex
import sys
from datetime import datetime, timezone
from pathlib import Path

entry = Path(LOCAL_1307_DIR) / ENTRY_SCRIPT
if not entry.exists():
    raise FileNotFoundError(
        f"ENTRY_SCRIPT not found: {entry}\n"
        "Update ENTRY_SCRIPT in the config cell."
    )

cmd = [sys.executable, str(entry)]
args = ENTRY_ARGS.strip()

# Auto-build required doe_geoai.py args when ENTRY_ARGS is left blank.
if not args and entry.name == "doe_geoai.py":
    if not DOE_DATASET_PATH.strip():
        raise ValueError(
            "doe_geoai.py needs -d/--dataset. "
            "Set DOE_DATASET_PATH in config or run the dataset-build step."
        )

    dataset = DOE_DATASET_PATH.strip()
    labelbin = DOE_LABELBIN_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_labels.l")
    model = DOE_MODEL_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_model.h5")
    plot = DOE_PLOT_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_plot.png")
    curves = DOE_CURVES_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_curves.csv")

    auto_args = [
        "-d", dataset,
        "-l", labelbin,
        "-m", model,
        "-p", plot,
        "-o", curves,
        "-e", str(DOE_EPOCHS),
        "-b", str(DOE_BATCH_SIZE),
        "-g", str(DOE_GPUS),
        "-k", str(DOE_KERNEL_PIXELS),
        "-c", str(DOE_CHANNELS),
    ]
    extra = DOE_EXTRA_ARGS.strip()
    if extra:
        auto_args.extend(shlex.split(extra))

    cmd.extend(auto_args)
else:
    if args:
        cmd.extend(shlex.split(args))

if AUTO_APPEND_OUTPUT_ARGS:
    cmd.extend([
        OUTPUT_DIR_FLAG,
        str(LOCAL_RUN_DIR),
        SAVE_DIR_FLAG,
        str(LOCAL_RUN_DIR / "checkpoints"),
    ])

# Write a reproducible run manifest before execution.
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "entry_script": ENTRY_SCRIPT,
    "entry_args": ENTRY_ARGS,
    "command": cmd,
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "gcp_project": GCP_PROJECT,
    "gcs_bucket": GCS_BUCKET,
    "gcs_code_prefix_1307": GCS_PREFIX_1307,
    "gcs_output_prefix": GCS_OUTPUT_PREFIX,
    "gcs_run_uri": GCS_RUN_URI,
    "local_code_dir": str(LOCAL_1307_DIR),
    "local_run_dir": str(LOCAL_RUN_DIR),
    "auto_append_output_args": AUTO_APPEND_OUTPUT_ARGS,
    "output_dir_flag": OUTPUT_DIR_FLAG,
    "save_dir_flag": SAVE_DIR_FLAG,
    "doe_build_dataset": DOE_BUILD_DATASET,
    "doe_gri_input": DOE_GRI_INPUT,
    "doe_dataset_out_dir": DOE_DATASET_OUT_DIR,
    "doe_channels": DOE_CHANNELS,
    "doe_sample_count": DOE_SAMPLE_COUNT,
    "doe_kernel_pixels": DOE_KERNEL_PIXELS,
    "doe_dataset_path": DOE_DATASET_PATH,
    "doe_labelbin_path": DOE_LABELBIN_PATH,
    "doe_model_path": DOE_MODEL_PATH,
    "doe_plot_path": DOE_PLOT_PATH,
    "doe_curves_path": DOE_CURVES_PATH,
    "doe_epochs": DOE_EPOCHS,
    "doe_batch_size": DOE_BATCH_SIZE,
    "doe_gpus": DOE_GPUS,
    "doe_extra_args": DOE_EXTRA_ARGS,
}
manifest_path = Path(LOCAL_RUN_DIR) / "run_config.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote", manifest_path)

run(cmd, cwd=LOCAL_1307_DIR)

## 9) Sync outputs to GCS

Run this during training and after training to persist checkpoints/logs/results.

If step 8 stops with missing DOE dataset path, set `DOE_BUILD_DATASET=True` and `DOE_GRI_INPUT` in config, then run step 7 to build it.

In [ ]:
from pathlib import Path

if "LOCAL_RUN_DIR" not in globals() or "GCS_RUN_URI" not in globals():
    raise RuntimeError("Run the persistent run directory cell first.")

if not Path(LOCAL_RUN_DIR).exists():
    raise FileNotFoundError(f"Missing local run directory: {LOCAL_RUN_DIR}")

run(["gsutil", "-m", "rsync", "-r", str(LOCAL_RUN_DIR), GCS_RUN_URI])
print("Synced:", GCS_RUN_URI)